In [ ]:
import os
os.chdir("..")

In [ ]:
from src.data.load_data import load_data
from src.data.preprocessing import clean_data
import numpy as np 
import pandas as pd

In [ ]:
df_raw = load_data("bitcoin_dataset.csv")
df = clean_data(df_raw)
df.head()

## Structural Hypothesis Testing: Price → Hash Rate Dynamics
We now move beyond visual observations to quantitatively examine the relationship between price and hash rate.
Based on mining economics and the patterns observed in the EDA section, we expect this relationship to be structurally defined and time‑lagged.
The purpose of this section is to formally analyze and test these dynamics.

### Checking Stationarity
We are going to examine whether the Price and Hash Rate time series are stationary.
This step is essential because the choice between VAR and VECM depends directly on the **stationarity** and **cointegration** properties of the data.

In [ ]:
from src.models.stationarity_tests import check_stationarity

df['log_price'] = np.log(df['btc_market_price'])
df['log_hashrate'] = np.log(df['btc_hash_rate'])
df.dropna(inplace=True)

check_stationarity(df['log_price'], 'log_price')
check_stationarity(df['log_hashrate'], 'log_hashrate')


In [ ]:
from src.models.var_model import select_var_lag
lag_results = select_var_lag(df[['log_price', 'log_hashrate']], 15)

Based on the results of the VAR lag order selection test, four standard criteria (AIC, BIC, FPE, and HQIC) were evaluated. The output showed that AIC and FPE reached their minimum values at lag 13, BIC at lag 4, and HQIC at lag 7. Since BIC typically favors more parsimonious models and reduces the risk of overfitting, **lag 4** was selected as the optimal choice for the VAR model. This result indicates that in the dynamic relationship between Bitcoin price and hash rate, their mutual effects are meaningfully reflected within a maximum delay of four time periods. This selected lag is also used to determine the number of differences **(p=3)** required for the Johansen cointegration test.
from statsmodels.tsa.vector_ar.vecm import coint_johansen

In [ ]:
from src.models.johansen import run_johansen_test
jres = run_johansen_test(df[['log_price', 'log_hashrate']], det_order=0, k_ar_diff=5)

Under the null hypothesis \( r <= 0 \), the **Trace statistic** is greater than the critical values. Therefore, the null hypothesis is rejected, indicating that **at least one cointegrating relationship** exists among the variables.
Under the hypothesis \( r <= 1 \), the null hypothesis is not rejected, as the **Trace statistic** is smaller than the critical values. Therefore, it can be concluded that **there is no more than one cointegrating relationship**, implying that **exactly one cointegrating relationship** exists between the variables.

In [ ]:
from src.models.vecm_model import fit_vecm
vecm_res = fit_vecm(df[['log_price', 'log_hashrate']], k_ar_diff=4, coint_rank=1)

The estimation results of the VECM model examining the relationship between the logarithm of price (log_price) and the logarithm of hash rate (log_hashrate) indicate that the dynamics between these variables can be separated into **short‑run dynamics** and a **long‑run equilibrium relationship**.

### Short‑Run Dynamics

In the equation for changes in price (Δlog_price), the coefficient of the first lag of price is −0.0534 and is statistically significant (p = 0.005). This result suggests that price changes exhibit short‑run adjustment behavior; in other words, an increase in price in the previous period is partially followed by a decrease in the current period’s price change. In contrast, the coefficients of lagged hash rate in this equation are not statistically significant, indicating that short‑term changes in hash rate do not have a significant effect on price changes in the short run. Similarly, the second and third lags of both price and hash rate are not statistically significant in this equation.

In the equation for changes in hash rate (Δlog_hashrate), the coefficients of lagged hash rate for the first through third lags are negative and statistically significant (p < 0.001). These results indicate that hash rate changes exhibit strong internal dynamics, meaning that past values of hash rate play an important role in determining its current changes. In contrast, the lagged values of price in this equation are generally not statistically significant, although the second lag of price is close to the significance threshold (p ≈ 0.056), suggesting a relatively weak short‑run effect of price on hash rate changes.

### Error Correction Mechanism

The loading coefficients (α) represent the speed at which variables adjust back to the long‑run equilibrium. The error correction coefficient in the price equation is 0.0017 and statistically significant (p < 0.001). This indicates that price responds to deviations from the long‑run equilibrium relationship, but the adjustment speed is very small. In the hash rate equation, the error correction coefficient is 0.0077 and statistically significant (p < 0.001), suggesting that hash rate reacts more strongly to deviations from equilibrium. Therefore, hash rate plays the **dominant role in restoring the system to its long‑run equilibrium**.

### Long‑Run Cointegration Relationship

The long‑run equilibrium relationship between the variables is given by the estimated cointegration equation:

log_price = 0.6343 log_hashrate − 3.2388

The coefficient of hash rate in this relationship is positive and statistically significant (p < 0.001), indicating that in the long run an increase in hash rate is associated with an increase in price. This finding confirms the existence of a **stable long‑run equilibrium relationship** between price and hash rate, implying that the two variables tend to move together around a common equilibrium path over time.

### Summary

Overall, the VECM results indicate that while short‑run interactions between price and hash rate are relatively limited, a stable long‑run equilibrium relationship exists between the two variables. Moreover, the adjustment coefficients suggest that **hash rate contributes more strongly than price to correcting deviations from the long‑run equilibrium**, highlighting its important role in the long‑term dynamics of the Bitcoin market.